In [38]:
library(GenomicRanges)
library(tidyr)
library(stringr)
library(locuszoomr)
library(dplyr)
library(cowplot)
library(ggplot2)
library(BiocGenerics)
library(GenomeInfoDb)
library(Seurat)

In [41]:
library(ggplot2)
library(Matrix)
library(ArchR)
library(schard)
library(ggrepel)

cellannotation="final_annotation"
proj_path="/nfs/team292/projects/PanTissue/results/freeze/ATAC/nonfetal_all/ArchRproj_subset/"

cells_fname=paste0(proj_path,"genefiles/cell_metadata.csv")
peaks_fname=paste0(proj_path,"genefiles/peak_matrix.rds")
genes_fname=paste0(proj_path,"genefiles/geneintegration_matrix.rds")

knn_atac_fname=paste0(proj_path,"Peak2GeneLinks/seATAC-Group-KNN.rds")
knn_rna_fname=paste0(proj_path,"Peak2GeneLinks/seRNA-Group-KNN.rds")

#adata_fname="/lustre/scratch125/cellgen/vento/cc53/trait2cell/ovary/scATAC/data/scRNAseq_forArchR_integration/ovaries_pediatric_rawcounts_allgenes.h5ad"

gene_files_outdir = paste0(proj_path,"/genefiles/")
plots_outdir=paste0(proj_path,"/Plots/")

In [42]:
loci_fname="/nfs/team292/projects/PanTissue/data/freeze/ATAC/GWAS_loci/GWAS_indexSNPs_hg38.csv"
OTAR_fname="/nfs/team292/projects/PanTissue/data/freeze/ATAC/GWAS_loci/repro_credible_sets.csv"
peaks2genes_fname=paste0(proj_path,"genefiles/peak2gene_links.csv")
celltypepeaks_dir=paste0(proj_path,"PeakCalls/peakcalling_annot")


In [43]:
geneAnnotation=readRDS("/nfs/team292/projects/PanTissue/data/temp/GWASxRNA/RNA_references/10x_ref_geneAnnotation_hg38_2020A.rds")

In [44]:
OTAR=read.csv(OTAR_fname,header=T)
loci=read.csv(loci_fname,header=T)
peaks2genes=read.csv(peaks2genes_fname,header=T)

In [62]:
peaks2genes[peaks2genes$peak=="chr2:120388859-120389359",]

,Correlation,FDR,VarQATAC,VarQRNA,geneName,chr,start,end,peak
,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
236418,0.8327127,2.351895e-21,0.7516343,0.9536663,INHBB,chr2,120388859,120389359,chr2:120388859-120389359


In [45]:
# build lookup from geneAnnotation
ensg_to_symbol <- setNames(geneAnnotation$gene$gene_name, geneAnnotation$gene$gene_id)
# replace l2g_top_geneId
OTAR$l2g_top_geneId <- ensg_to_symbol[OTAR$l2g_top_geneId]
# replace l2g_genes (comma-separated)
OTAR$l2g_genes <- sapply(OTAR$l2g_genes, function(x) {
    if (is.na(x)) return(NA)
    genes <- strsplit(x, ", ")[[1]]
    paste(ensg_to_symbol[genes], collapse=", ")
})

In [46]:
OTAR_fixed=OTAR[,c('studyId','variantId','chromosome','position','pValue','traitFromSource','publicationFirstAuthor','publicationDate','diseaseName',
				'l2g_top_geneId','l2g_top_score','l2g_genes','l2g_scores')]
loci_fixed=loci %>% dplyr::rename(position=Position_hg38_lifted,interval_start=Interval_start_bp_hg38,interval_end=Interval_end_bp_hg38,variantId=variantID) %>% 
	mutate(traitFromSource=diseaseName) %>%
	select(-c(Position_hg38,Interval_length_bp,Position_hg19,Interval_start_bp_hg19,Interval_end_bp_hg19))

In [47]:
OTAR_fixed$chromosome <- as.integer(OTAR_fixed$chromosome)
loci_fixed$chromosome <- as.integer(loci_fixed$chromosome)
OTAR_fixed$publicationDate <- as.character(OTAR_fixed$publicationDate)
loci_fixed$publicationDate <- as.character(loci_fixed$publicationDate)
allloci=dplyr::bind_rows(OTAR_fixed,loci_fixed)

Warning message in eval(expr, envir, enclos):
"NAs introduced by coercion"


In [48]:
allloci <- allloci %>%
  distinct(publicationFirstAuthor, traitFromSource, diseaseName, chromosome, position, .keep_all = TRUE)

# fix up my peaks

In [49]:
peaks2genes = peaks2genes %>% mutate(peak=peakName) %>%
								separate(peakName, into=c("chr","coords"),sep=":") %>%
								separate(coords, into=c("start","end"),sep="-") %>%
								#mutate(CHR=gsub("chr","",chr)) %>% 
								select(-idxATAC,-idxRNA)
head(peaks2genes)

,Correlation,FDR,VarQATAC,VarQRNA,geneName,chr,start,end,peak
,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
1,0.6005109,2.656229e-08,0.9668777,0.2652881,FAM87B,chr1,905195,905695,chr1:905195-905695
2,0.5930539,4.643301e-08,0.9276664,0.2652881,FAM87B,chr1,980609,981109,chr1:980609-981109
3,0.5273422,3.493845e-06,0.6809746,0.2652881,FAM87B,chr1,1014969,1015469,chr1:1014969-1015469
4,0.4967756,1.899876e-05,0.7154563,0.2652881,FAM87B,chr1,1124095,1124595,chr1:1124095-1124595
5,0.4703327,7.148032e-05,0.5324975,0.7211730,LINC01128,chr1,807763,808263,chr1:807763-808263
6,0.4858853,3.327975e-05,0.5551324,0.7211730,LINC01128,chr1,830678,831178,chr1:830678-831178


In [50]:
ranges <- GRanges(seqnames = peaks2genes$chr, ranges = IRanges(start = as.numeric(peaks2genes$start), end = as.numeric(peaks2genes$end)),range_id = peaks2genes$peak)

In [51]:
snps_gr <- GRanges(
  seqnames = paste0("chr", allloci$chromosome),
  ranges = IRanges(start = allloci$position, end = allloci$position),
  range_id=allloci$variantId)

In [52]:
common <- intersect(seqlevels(snps_gr), seqlevels(ranges))
snps_gr <- keepSeqlevels(snps_gr, common,pruning.mode = "coarse")
ranges <- keepSeqlevels(ranges, common,pruning.mode = "coarse")

In [53]:
hits <- findOverlaps(snps_gr, ranges)

In [54]:
disease_hits <- data.frame(rsID  = snps_gr$range_id[queryHits(hits)],peak   = ranges$range_id[subjectHits(hits)]) %>% distinct()
head(disease_hits)
dim(disease_hits)

,rsID,peak
,<chr>,<chr>
1,1_1351675_G_A,chr1:1351362-1351862
2,1_7849474_G_A,chr1:7849290-7849790
3,1_8449353_C_T,chr1:8449137-8449637
4,1_9290380_C_T,chr1:9290073-9290573
5,1_12133042_C_T,chr1:12132928-12133428
6,1_15195861_C_T,chr1:15195611-15196111


[1] 461   2

In [55]:
paste0("Number of peaks linked to a gene and overlapping a GWAS locus: ",length(unique(disease_hits$peak)))

[1] "Number of peaks linked to a gene and overlapping a GWAS locus: 450"

In [56]:
head(disease_hits)

,rsID,peak
,<chr>,<chr>
1,1_1351675_G_A,chr1:1351362-1351862
2,1_7849474_G_A,chr1:7849290-7849790
3,1_8449353_C_T,chr1:8449137-8449637
4,1_9290380_C_T,chr1:9290073-9290573
5,1_12133042_C_T,chr1:12132928-12133428
6,1_15195861_C_T,chr1:15195611-15196111


In [63]:
disease_hits[disease_hits$peak=="chr2:120388859-120389359",]

,rsID,peak
,<chr>,<chr>
274,2_120388925_T_C,chr2:120388859-120389359


In [18]:
#format disease_hits with corresponding GWAS, correlation and other info
rows_list=list()
list_idx=1
for (row in 1:nrow(disease_hits)){
    rsID=disease_hits$rsID[row]
    loci_sub=allloci[allloci$variantId==rsID,]

    peak=disease_hits$peak[row]
    peaks2genes_sub=peaks2genes[peaks2genes$peak==peak,]

    peak_info="_gene
	_gene"
    if (nrow(peaks2genes_sub) > 1) {
    	max_cor  <- max(peaks2genes_sub$Correlation, na.rm = TRUE)
    	min_fdr  <- min(peaks2genes_sub$FDR, na.rm = TRUE)
    
    	genes    <- paste(peaks2genes_sub$geneName, collapse = ",")  # all genes
    
    	top_sub  <- peaks2genes_sub[peaks2genes_sub$Correlation == max_cor, ]
    	peak_info <- "multi_genes"
    	if (any(top_sub$FDR != min_fdr)) next
    	if (nrow(top_sub) > 1) top_sub <- top_sub[1, ]
    	topgene  <- top_sub$geneName
    	peaks2genes_sub <- top_sub
	} else {
    	genes    <- peaks2genes_sub$geneName
    	topgene  <- peaks2genes_sub$geneName
    	peak_info <- "unique_gene"
	}

	peaks2genes_sub$topgene <- topgene
	peaks2genes_sub$genes   <- genes

    for (gwas in unique(loci_sub$traitFromSource[!is.na(loci_sub$traitFromSource)])){
        rows_list[[list_idx]]= disease_hits[row,] %>% 
            left_join(loci_sub[loci_sub$traitFromSource==gwas,], by=c("rsID" = "variantId")) %>% 
            left_join(peaks2genes_sub, by="peak") %>% 
            dplyr::mutate(peak_info=peak_info)
        list_idx=list_idx+1
    }
}

finalres=do.call(rbind, rows_list)

In [19]:
#remove peaks inside of their top gene
# build GRanges from finalres peaks
finalres$start=as.numeric(finalres$start)
finalres$end=as.numeric(finalres$end)
peaks_gr <- GRanges(
    seqnames = finalres$chr,
    ranges = IRanges(start = finalres$start, end = finalres$end)
)

# expand gene annotation by 1000bp
gene_gr <- geneAnnotation$gene
start(gene_gr) <- pmax(1, start(gene_gr))#- 1000
end(gene_gr) <- end(gene_gr)# + 1000

# subset gene annotation to only genes in finalres
gene_gr <- gene_gr[gene_gr$gene_name %in% finalres$geneName]

# find which peaks overlap their own gene
hits <- findOverlaps(peaks_gr, gene_gr)

# for each hit, check if the gene name matches
overlapping_same_gene <- mapply(function(q, s) {
    finalres$geneName[q] == gene_gr$gene_name[s]
}, queryHits(hits), subjectHits(hits))

# rows where peak overlaps its own gene
proximal_rows <- queryHits(hits)[overlapping_same_gene]

# keep only distal (not overlapping own gene)
finalres_distal <- finalres[-unique(proximal_rows), ]

In [20]:
finalres_distal_conf=finalres_distal %>% filter(FDR<1e-5)

In [27]:
unique(finalres_distal_conf$publicationFirstAuthor)

[1] "Fehringer G"      "Pujol Gualdo N"   "Kar SP"           "Shigesi N"       
 [5] "Kim J"            "Sakaue S"         "Verma A"          ""                
 [9] "Rahmioglu N"      "Kuchenbaecker KB" "Sliz E"           "Edwards TL"      
[13] "Loya H"           "Tyrmi JS"         "Ramachandran D"   "Venkatesh SS"    
[17] "Gallagher CS"     "Kiewa J"          "Rafnar T"         "Wu X"            
[21] "Chen MM"          "Xiao C"           "Koel M"           "Wang S"          
[25] "Jiang L"          "Lee CC"           "Rashkin SR"       "Dareng EO"       
[29] "Goode EL"         "Pharoah PD"       "Phelan CM"        "Guo H"           
[33] "Sato G"           "O'Mara TA"        "Pathare ADS"      "Zhao X"          
[37] "Moolhuijsen LME"

In [21]:
head(finalres_distal_conf)

,rsID,peak,studyId,chromosome,position,pValue,traitFromSource,publicationFirstAuthor,publicationDate,diseaseName,⋯,FDR,VarQATAC,VarQRNA,geneName,chr,start,end,topgene,genes,peak_info
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1_1351675_G_A,chr1:1351362-1351862,GCST003588,1,1351675,5.000000e-10,Cancer (pleiotropy),Fehringer G,2016-04-20,ovarian endometrioid carcinoma,⋯,4.961032e-10,0.8445309,0.9783549,MXRA8,chr1,1351362,1351862,MXRA8,"SDF4,MXRA8,TMEM240",multi_genes
2,1_1351675_G_A,chr1:1351362-1351862,GCST003588,1,1351675,5.000000e-10,Cancer (pleiotropy),Fehringer G,2016-04-20,ovarian carcinoma,⋯,4.961032e-10,0.8445309,0.9783549,MXRA8,chr1,1351362,1351862,MXRA8,"SDF4,MXRA8,TMEM240",multi_genes
3,1_1351675_G_A,chr1:1351362-1351862,GCST003588,1,1351675,5.000000e-10,Cancer (pleiotropy),Fehringer G,2016-04-20,ovarian serous carcinoma,⋯,4.961032e-10,0.8445309,0.9783549,MXRA8,chr1,1351362,1351862,MXRA8,"SDF4,MXRA8,TMEM240",multi_genes
4,1_7849474_G_A,chr1:7849290-7849790,GCST90454206,1,7849474,1.130048e-17,ICD10 E28.3: Primary ovarian failure,Pujol Gualdo N,2025-03-11,ovarian dysfunction,⋯,1.681852e-07,0.3488987,0.5598441,AL034417.3,chr1,7849290,7849790,AL034417.3,"AL034417.3,AL034417.2",multi_genes
5,1_8449353_C_T,chr1:8449137-8449637,GCST90454240,1,8449353,7.030000e-11,ICD10 O45: Premature separation of placenta (abruptio placentae),Pujol Gualdo N,2025-03-11,Abruptio Placentae,⋯,6.338344e-08,0.8816063,0.5890267,SLC45A1,chr1,8449137,8449637,SLC45A1,SLC45A1,unique_gene
6,1_9290380_C_T,chr1:9290073-9290573,GCST90454206,1,9290380,1.830000e-13,ICD10 E28.3: Primary ovarian failure,Pujol Gualdo N,2025-03-11,ovarian dysfunction,⋯,7.295671e-10,0.9432340,0.9081006,SPSB1,chr1,9290073,9290573,SPSB1,"SPSB1,SLC25A33",multi_genes


In [64]:
finalres_distal_conf[finalres_distal_conf$peak=="chr2:120388859-120389359",]

,rsID,peak,studyId,chromosome,position,pValue,traitFromSource,publicationFirstAuthor,publicationDate,diseaseName,⋯,FDR,VarQATAC,VarQRNA,geneName,chr,start,end,topgene,genes,peak_info
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
213,2_120388925_T_C,chr2:120388859-120389359,FINNGEN_R12_E4_PCOS_BROAD,2,120388925,3.744e-08,"Polycystic ovarian syndrome, broad definition",,,polycystic ovary syndrome,⋯,2.351895e-21,0.7516343,0.9536663,INHBB,chr2,120388859,120389359,INHBB,INHBB,unique_gene


In [32]:
row <- finalres_distal_conf[finalres_distal_conf$publicationFirstAuthor == "Moolhuijsen LME", ]
row[1, ]  # check what the row looks like
nrow(row) # check it found rows at all
colnames(row)[is.na(row[1, ])]


,rsID,peak,studyId,chromosome,position,pValue,traitFromSource,publicationFirstAuthor,publicationDate,diseaseName,⋯,FDR,VarQATAC,VarQRNA,geneName,chr,start,end,topgene,genes,peak_info
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
347,rs804263,chr8:11773077-11773577,NA,8,11773406,NA,polycystic ovary syndrome,Moolhuijsen LME,2026,polycystic ovary syndrome,⋯,1.019437e-08,0.274219,0.8230275,FAM167A-AS1,chr8,11773077,11773577,FAM167A-AS1,"FAM167A-AS1,FAM167A",multi_genes


[1] 3

[1] "studyId"        "pValue"         "l2g_top_geneId" "l2g_top_score" 
[5] "l2g_genes"      "l2g_scores"

In [33]:
first_nonNA <- function(x) {
  val <- x[!is.na(x)]
  if (length(val) == 0) NA else val[1]
}

agg_res <- finalres_distal_conf %>%
  mutate(
    pub_entry = case_when(
      !is.na(studyId) & str_detect(studyId, "FINNGEN") ~ paste(traitFromSource, "FinnGen", sep = ";"),
      TRUE ~ paste(traitFromSource, publicationFirstAuthor, publicationDate, sep = ";")
    )
  ) %>%
  group_by(peak) %>%
  dplyr::summarise(
    peak           = first(peak),
    diseaseName    = paste(unique(na.omit(diseaseName)), collapse = " | "),
    publications   = paste(unique(na.omit(pub_entry)),   collapse = " | "),
    pValue         = first_nonNA(pValue),
    l2g_top_geneId = first_nonNA(l2g_top_geneId),
    l2g_top_score  = first_nonNA(l2g_top_score),
    l2g_genes      = first_nonNA(l2g_genes),
    l2g_scores     = first_nonNA(l2g_scores),
    Correlation    = first_nonNA(Correlation),
    FDR            = first_nonNA(FDR),
    geneName       = first_nonNA(geneName),
    genes          = first_nonNA(genes),
    peak_info      = first_nonNA(peak_info)
  ) %>%
  ungroup() %>%
  mutate(genes = str_replace_all(genes, ",", "|")) %>%
  dplyr::mutate(across(where(is.character), ~ str_replace_all(.x, ",", "")))

In [34]:
head(agg_res)

peak,diseaseName,publications,pValue,l2g_top_geneId,l2g_top_score,l2g_genes,l2g_scores,Correlation,FDR,geneName,genes,peak_info
<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
chr10:102652524-102653024,pyelonephritis,Pyelonephritis (PheCode 590);Verma A;2024-07-19,2.870000e-13,TRIM8,0.4845357,TRIM8,0.48453572392463684,0.6362683,1.460179e-09,WBP1L,MFSD13A|WBP1L,multi_genes
chr10:102727345-102727845,Uterine leiomyoma,Uterine leiomyomata;Sliz E;2023-02-01 | ICD10 D25: Leiomyoma of uterus;Pujol Gualdo N;2025-03-11,2.377376e-08,SFXN2,0.4260618,SFXN2,0.42606180906295776,0.6081797,1.471884e-08,MFSD13A,MFSD13A|WBP1L,multi_genes
chr10:103923534-103924034,uterine fibroid,Uterine fibroids;Sakaue S;2021-09-30,3.219700e-12,STN1,0.8827955,STN1 SLK,0.882795512676239 0.0540400929749012,0.8575800,5.370811e-24,CFAP43,CFAP43|CFAP58,multi_genes
chr10:110151454-110151954,ovarian dysfunction,ICD10 E28.3: Primary ovarian failure;Pujol Gualdo N;2025-03-11,1.680000e-09,MXI1,0.2959977,MXI1 ADD3,0.295997679233551 0.08423151820898056,0.5113038,8.693138e-06,DUSP5,DUSP5,unique_gene
chr10:132775106-132775606,placental retention,ICD10 O73: Retained placenta and membranes without haemorrhage;Pujol Gualdo N;2025-03-11,5.630000e-10,INPP5A,0.4805146,INPP5A NKX6-2,0.4805145561695099 0.16516005992889404,0.6088736,1.394515e-08,ADGRA1,LRRC27|PWWP2B|AL451069.2|CFAP46|LINC01166|ADGRA1|KNDC1,multi_genes
chr10:17178797-17179297,cervical carcinoma,ICD10 C53: Malignant neoplasm of cervix uteri;Pujol Gualdo N;2025-03-11,1.420000e-09,TRDMT1,0.8273166,TRDMT1 CUBN VIM,0.8273165822029114 0.10979399085044861 0.10688824206590652,0.7310693,6.596573e-14,AL133415.1,VIM-AS1|AL133415.1,multi_genes


In [35]:
paste0("Number of peaks linked to a gene and overlapping a GWAS locus: ",length(unique(agg_res$peak)))

[1] "Number of peaks linked to a gene and overlapping a GWAS locus: 290"

In [36]:
n=agg_res %>%
  pull(diseaseName) %>%
  str_split(" \\| ") %>%
  unlist() %>%
  unique() %>%
  length()
paste0("Number of diseases with a peak-gene link: ",n)

[1] "Number of diseases with a peak-gene link: 36"

In [37]:
write.csv(agg_res,"/nfs/team292/projects/PanTissue/results/freeze/ATAC/peak2gene/finalres/disease_links_FDR1e-5_uniquepeaks.csv",quote=F,row.names=F)

In [24]:
finalres_distal_conf_uniquegene=finalres_distal_conf[finalres_distal_conf$peak_info=='unique_gene',]

In [25]:
paste0("Number of peaks linked to a gene and overlapping a GWAS locus: ",length(unique(finalres_distal_conf_uniquegene$peak)))

[1] "Number of peaks linked to a gene and overlapping a GWAS locus: 94"

In [ ]:
write.csv(finalres_distal_conf_uniquegene,"/nfs/team292/projects/PanTissue/results/freeze/ATAC/peak2gene/finalres/disease_links_FDR1e-5_uniquegene.csv",quote=F,row.names=F)

In [26]:
agg_res <- finalres_distal_conf_uniquegene %>%
  mutate(
    pub_entry = case_when(
      !is.na(studyId) & str_detect(studyId, "FINNGEN") ~ paste(traitFromSource, "FinnGen", sep = ";"),
      TRUE ~ paste(traitFromSource, publicationFirstAuthor, publicationDate, sep = ";")
    )
  ) %>%
  group_by(peak) %>%
  dplyr::summarise(
    peak           = first(peak),
    diseaseName    = paste(unique(na.omit(diseaseName)), collapse = " | "),
    publications   = paste(unique(na.omit(pub_entry)),   collapse = " | "),
    l2g_top_geneId = first(na.omit(l2g_top_geneId)),
    l2g_top_score  = first(na.omit(l2g_top_score)),
    Correlation    = first(na.omit(Correlation)),
    FDR            = first(na.omit(FDR)),
    geneName       = first(na.omit(geneName))
  ) %>%
  ungroup() %>%
  dplyr::mutate(across(where(is.character), ~ str_replace_all(.x, ",", "")))

Warning message:
"Returning more (or less) than 1 row per `summarise()` group was deprecated in dplyr 1.1.0.
ℹ Please use `reframe()` instead.
ℹ When switching from `summarise()` to `reframe()`, remember that `reframe()` always returns an ungrouped data frame and adjust accordingly."
`summarise()` has grouped output by 'peak'. You can override using the `.groups` argument.


In [ ]:
write.csv(agg_res,"/nfs/team292/projects/PanTissue/results/freeze/ATAC/peak2gene/finalres/disease_links_FDR1e-5_uniquegene_uniquepeaks.csv",quote=F,row.names=F)

# plot

In [ ]:
# Genes per peak
genes_per_peak <- as.data.frame(table(peaks2genes$peak))
colnames(genes_per_peak) <- c("peak", "n_genes")

single_gene_peaks <- sum(genes_per_peak$n_genes == 1)
proportion <- single_gene_peaks / nrow(genes_per_peak)
paste0("Fraction of peaks linked to only 1 gene: ",round(proportion,3))

p1=ggplot(data=genes_per_peak,aes(x=n_genes))+
	geom_histogram(binwidth = 1, fill = "darkcyan")+
	labs(x="Number of genes",y="Number of peaks")+
	theme_classic()

# Peaks per gene
peaks_per_gene <- as.data.frame(table(peaks2genes$geneName))
colnames(peaks_per_gene) <- c("gene", "n_peaks")

single_peak_genes <- sum(peaks_per_gene$n_peaks == 1)
proportion <- single_peak_genes / nrow(peaks_per_gene)
paste0("Fraction of genes linked to only 1 peak: ",round(proportion,3))

p2=ggplot(data=peaks_per_gene,aes(x=n_peaks))+
	geom_histogram(binwidth = 1, fill = "chocolate3")+
	labs(x="Number of peaks",y="Number of genes")+
	theme_classic()

pdf(paste0(plots_outdir,"peak2genelinks_histograms.pdf"), width = 2, height = 2)
p1
p2
dev.off()

[1] "Fraction of peaks linked to only 1 gene: 0.499"

[1] "Fraction of genes linked to only 1 peak: 0.091"

png 
  2